# 📊 Employee Performance Predictor — EDA Notebook
> **Purpose:** Exploratory Data Analysis to understand the HR dataset before modelling.
> Run `python main.py` first to generate the dataset in `../data/hr_dataset.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
# If dataset doesn't exist, generate it
import os
if not os.path.exists('../data/hr_dataset.csv'):
    from src.data_generator import generate_dataset
    df = generate_dataset(n_employees=500, save=True, output_dir='../data')
else:
    df = pd.read_csv('../data/hr_dataset.csv')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
df.describe().round(2)

## 2. Target Variable — Performance Band Distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Count
counts = df['performance_band'].value_counts().reindex(['High','Medium','Low'])
colors = ['#2ecc71', '#f39c12', '#e74c3c']
ax1.bar(counts.index, counts.values, color=colors, width=0.5, edgecolor='white')
ax1.set_title('Performance Band — Count', fontweight='bold')
ax1.set_ylabel('Count')

# Pie
ax2.pie(counts.values, labels=counts.index, colors=colors,
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Performance Band — Proportion', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Feature Distributions by Band

In [ ]:
features = ['training_hours', 'attendance_rate', 'goal_achievement_pct',
            'manager_rating', 'peer_score', 'overtime_hours']
palette  = {'High': '#2ecc71', 'Medium': '#f39c12', 'Low': '#e74c3c'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, feat in zip(axes, features):
    for band, color in palette.items():
        subset = df[df['performance_band'] == band][feat].dropna()
        ax.hist(subset, bins=20, alpha=0.55, color=color, label=band, density=True)
    ax.set_title(feat.replace('_',' ').title(), fontsize=11, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Performance Band', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

In [ ]:
num_cols = ['training_hours', 'attendance_rate', 'goal_achievement_pct',
            'manager_rating', 'peer_score', 'overtime_hours',
            'experience_years', 'salary', 'projects_completed', 'performance_score']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size':8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Quarterly Performance Trend

In [ ]:
def q_key(s):
    y, q = s.split('-Q')
    return int(y)*10 + int(q)

trend = df.groupby('quarter_label')['performance_score'].mean().reset_index()
trend['_k'] = trend['quarter_label'].apply(q_key)
trend = trend.sort_values('_k')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(trend['quarter_label'], trend['performance_score'],
        marker='o', color='#3498db', linewidth=2)
ax.fill_between(trend['quarter_label'], trend['performance_score'], alpha=0.15, color='#3498db')
ax.set_title('Average Performance Score by Quarter', fontweight='bold')
ax.set_ylabel('Avg Performance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Department Analysis

In [ ]:
dept_band = (df.groupby(['department','performance_band'])
               .size().reset_index(name='count'))
pivot = dept_band.pivot(index='department', columns='performance_band', values='count').fillna(0)
pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100
for col in ['High','Medium','Low']:
    if col not in pivot.columns: pivot[col] = 0
pivot = pivot[['High','Medium','Low']]

fig, ax = plt.subplots(figsize=(11, 5))
pivot.plot(kind='bar', ax=ax,
           color=['#2ecc71','#f39c12','#e74c3c'], edgecolor='white', width=0.65)
ax.set_title('Performance Band by Department (%)', fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.legend(title='Band')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 7. Key Insight: Training Hours vs Performance
> **Business Question:** Does investing in training actually improve performance?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
palette = {'High':'#2ecc71','Medium':'#f39c12','Low':'#e74c3c'}

for band, color in palette.items():
    sub = df[df['performance_band']==band]
    ax.scatter(sub['training_hours'], sub['performance_score'],
               alpha=0.3, s=15, color=color, label=band)

# Trend line
x = df['training_hours'].dropna()
y = df.loc[x.index,'performance_score'].dropna()
idx = x.index.intersection(y.index)
z = np.polyfit(x[idx], y[idx], 1)
p = np.poly1d(z)
xline = np.linspace(x.min(), x.max(), 100)
ax.plot(xline, p(xline), 'k--', linewidth=1.5, label=f'Trend (slope={z[0]:.2f})')

ax.set_title('Training Hours vs Performance Score', fontweight='bold')
ax.set_xlabel('Training Hours per Quarter')
ax.set_ylabel('Performance Score')
ax.legend()
plt.tight_layout()
plt.show()

corr_val = df[['training_hours','performance_score']].corr().iloc[0,1]
print(f'\nCorrelation: {corr_val:.3f}')
print('→ Positive correlation confirms: more training → higher performance')

## 8. Summary Statistics per Band

In [ ]:
summary = df.groupby('performance_band')[[
    'training_hours','attendance_rate','goal_achievement_pct',
    'manager_rating','peer_score','overtime_hours','experience_years'
]].mean().round(2)
summary

In [ ]:
print('\n✅ EDA Complete!')
print('Next step: Run main.py for full ML pipeline')
print('  > python main.py')